## Специалист по информационным технологиям

### Агент, отвечающий на вопросы, который является специалистом по информационным технологиям
### Будет использоваться сотрудниками Insurellm, страховой технологической компании
### Агент должен быть точным, а решение должно быть недорогим.

В этом проекте будет использоваться RAG (Расширенная генерация поиска), чтобы обеспечить высокую точность работы нашего помощника по вопросам/ответам.

In [1]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [2]:
# imports for langchain and Chroma and plotly

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [3]:
# цена является важным фактором для нашей компании, поэтому мы собираемся использовать недорогую модель

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [4]:
# Загрузить переменные среды в файл с именем .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [5]:
# Прочитайте документы с помощью загрузчиков LangChain
# Найдите все из всех вложенных папок нашей базы знаний

folders = glob.glob("knowledge-base/*")

# Выражаем благодарность CG и Jon R, слушателям курса, за это исправление, необходимое некоторым пользователям
text_loader_kwargs = {'encoding': 'utf-8'}
# Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

# Пожалуйста, обратите внимание:

В следующей ячейке мы разбили текст на фрагменты.

2 студента сообщили мне, что в следующей ячейке произошел сбой в работе их компьютера.  
Они смогли исправить это, изменив размер фрагмента с 1000 на 2000, а размер фрагмента с 200 на 400.  
Это не должно быть обязательным, но если это случится с вами, пожалуйста, внесите это изменение!  
((Обратите внимание, что длинная цепочка может выдавать предупреждение о том, что размер фрагмента больше 1000 - это можно смело игнорировать).

_Большое спасибо Стивену У. и NirP за этот ценный вклад._

In [6]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1088, which is longer than the specified 1000


In [7]:
len(chunks)

123

In [8]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: products, employees, company, contracts


## Небольшое замечание о встраиваниях и "Фильмах с автоматическим кодированием"

Мы будем отображать каждый фрагмент текста в вектор, который представляет значение текста, что называется встраиванием.

Open air предлагает модель для этого, которую мы будем использовать, вызывая их API с помощью некоторого длинного кода.

Эта модель является примером "LLM с автоматическим кодированием", которая генерирует выходные данные на основе полных входных данных.
Это отличается от всех других конечностей, которые мы обсуждали сегодня, которые известны как "авторегрессивные конечности" и генерируют будущие токены только на основе прошлого контекста.

Другим примером Lms с автоматическим кодированием является BERT от Google. Помимо встраивания, Lms с автоматическим кодированием часто используются для классификации.

### Sidenote

На восьмой неделе мы вернемся к RAG и векторным встраиваниям и будем использовать векторный кодировщик с открытым исходным кодом, чтобы данные никогда не покидали наш компьютер - это важный момент при создании корпоративных систем, и данные должны оставаться внутренними.

In [9]:
# Поместите фрагменты данных в векторное хранилище, которое связывает векторное вложение с каждым фрагментом

embeddings = OpenAIEmbeddings()

# Если вы предпочитаете использовать бесплатные векторные вставки из предложения Hugging Face-трансформеры
# Then replace embeddings = OpenAIEmbeddings()
# with:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [10]:
# Проверьте, существует ли хранилище данных Chroma - если да, удалите коллекцию, чтобы начать с нуля.

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [11]:
# Создайте наш магазин цветных векторов!

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 123 documents


In [12]:
# Get one vector and find how many dimensions it has

collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

The vectors have 1,536 dimensions


## Визуализация хранилища векторов

Давайте на минутку взглянем на документы и векторы для их встраивания, чтобы понять, что происходит.

In [13]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
doc_types = [metadata['doc_type'] for metadata in result['metadatas']]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [14]:
# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [15]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()